[source](https://scicomp.stackexchange.com/questions/37336/solving-numerically-the-1d-kuramoto-sivashinsky-equation-using-spectral-methods)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm

In [ ]:
nu = 1
L = 100 
nx = 1024

t0 = 0
tN = 300
dt = 0.05
nt = int((tN - t0) / 0.05)

# wave number mesh
k = np.arange(-nx/2, nx/2, 1)

t = np.linspace(start=t0, stop=tN, num=nt)
x = np.linspace(start=0, stop=L, num=nx)

# solution mesh in real space
u = np.ones((nx, nt))
# solution mesh in Fourier space
u_hat = np.ones((nx, nt), dtype=complex)

u_hat2 = np.ones((nx, nt), dtype=complex)

# initial condition 
u0 = np.cos((2 * np.pi * x) / L) + 0.1 * np.cos((4 * np.pi * x) / L)

# Fourier transform of initial condition
u0_hat = (1 / nx) * np.fft.fftshift(np.fft.fft(u0))

u0_hat2 = (1 / nx) * np.fft.fftshift(np.fft.fft(u0**2))

# set initial condition in real and Fourier mesh
u[:,0] = u0
u_hat[:,0] = u0_hat

u_hat2[:,0] = u0_hat2


In [ ]:

# Fourier Transform of the linear operator
FL = (((2 * np.pi) / L) * k) ** 2 - nu * (((2 * np.pi) / L) * k) ** 4
# Fourier Transform of the non-linear operator
FN = - (1 / 2) * ((1j) * ((2 * np.pi) / L) * k)

# resolve EDP in Fourier space
for j in range(0,nt-1):
  uhat_current = u_hat[:,j]
  uhat_current2 = u_hat2[:,j]
  if j == 0:
    uhat_last = u_hat[:,0]
    uhat_last2 = u_hat2[:,0]
  else:
    uhat_last = u_hat[:,j-1]
    uhat_last2 = u_hat2[:,j-1]
  
  # compute solution in Fourier space through a finite difference method
  # Cranck-Nicholson + Adam 
  u_hat[:,j+1] = (1 / (1 - (dt / 2) * FL)) * ( (1 + (dt / 2) * FL) * uhat_current + ( ((3 / 2) * FN) * (uhat_current2) - ((1 / 2) * FN) * (uhat_last2) ) * dt )
  # go back in real space
  u[:,j+1] = np.real(nx * np.fft.ifft(np.fft.ifftshift(u_hat[:,j+1])))
  u_hat2[:,j+1] = (1 / nx) * np.fft.fftshift(np.fft.fft(u[:,j+1]**2))


In [ ]:
# plot the result
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t)
cs = ax.contourf(xx, tt, u.T, cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Kuramoto-Sivashinsky: L = {L}, nu = {nu}")

In [ ]:
from IPython.display import clear_output
import time

def visualize(x, u, u2=None, frame_skip=100, max_frames=20, labels=('u1', 'u2')):
    n_time = u.shape[1]

    total_frames = min(n_time // frame_skip, max_frames)
    frames_to_show = np.linspace(0, n_time-1, total_frames, dtype=int)

    for frame in frames_to_show:
        clear_output(wait=True)

        plt.figure(figsize=(10, 6))
        plt.plot(x, u[:, frame], 'b-', linewidth=2, label=labels[0])
        if u2 is not None:
            plt.plot(x, u2[:, frame], 'r--', linewidth=2, label=labels[1])
            plt.legend()
        plt.xlabel('Position (x)')
        plt.ylabel('Amplitude (u)')
        plt.title(f'Frame {frame} ({(frame/(n_time-1)*100):.1f}%)')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        time.sleep(0.5)

In [ ]:
#visualize(x, u, 10, 200)

In [ ]:
u.shape

In [ ]:
data = u[:,2000:].T
#np.save("1dks.npy", data)

In order to determine proper timelag size, we will use autocorrelation function

In [ ]:
from scipy.signal import correlate

def correlation_time(data, dt):
    """data shape: (n_time, n_space)"""
    n_time, n_space = data.shape
    max_lag = n_time // 2

    autocorr = np.zeros(max_lag)
    for i in range(n_space):
        series = data[:, i] - data[:, i].mean()
        full = correlate(series, series, mode='full')
        full /= full[n_time - 1]  # normalize by zero-lag
        autocorr += full[n_time - 1:n_time - 1 + max_lag]
    autocorr /= n_space

    lags = np.arange(max_lag) * dt

    # find 1/e crossing
    tau_idx = np.argmax(autocorr < 1/np.e)
    tau = lags[tau_idx]

    plt.figure(figsize=(10, 4))
    plt.plot(lags, autocorr)
    plt.axhline(1/np.e, color='r', ls='--', label=f'1/e')
    plt.axvline(tau, color='g', ls='--', label=f'τ = {tau:.2f}s')
    plt.xlabel('Time lag (s)')
    plt.ylabel('Autocorrelation')
    plt.title('Spatially-averaged temporal autocorrelation')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xlim(0, min(lags[-1], tau * 5))
    plt.show()

    return tau

tau = correlation_time(data, dt=0.05)
print(f'Correlation time: {tau:.2f}s → lags ≈ {int(tau / 0.05)}')

In [ ]:
from pyshred import DataManager, SHRED, SHREDEngine

import pyshred
pyshred.set_default_device(device="cuda")

manager = DataManager(
    lags=125,#20 seconds
    train_size=0.8,
    val_size=0.1,
    test_size=0.1
)

In [ ]:
manager.add_data(
    data=data,
    id="1dks",
    random=20,
    compress=20
)

In [ ]:
train_dataset, val_dataset, test_dataset= manager.prepare()

In [ ]:
import pysindy as ps
from pyshred import SINDy_Forecaster, GRU, LSTM, MLP
latent_forecaster = SINDy_Forecaster(
    poly_order=1,
    include_sine=False,
    dt=0.05,
    optimizer=ps.STLSQ(threshold=1e-2, alpha=0.05)
)

shred = SHRED(
    sequence_model=GRU(hidden_size=16, num_layers=2),
    decoder_model=MLP(hidden_sizes=[256, 512], dropout=.1),
    latent_forecaster=latent_forecaster
)

In [ ]:
val_errors = shred.fit(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    lr=3e-4,
    num_epochs=50,
    sindy_regularization=5,
    sindy_thres_epoch=20,
    verbose=True
)

print('val_errors:', val_errors)


In [ ]:
train_mse = shred.evaluate(dataset=train_dataset)
val_mse = shred.evaluate(dataset=val_dataset)
test_mse = shred.evaluate(dataset=test_dataset)
print(f"Train MSE: {train_mse:.3f}")
print(f"Val   MSE: {val_mse:.3f}")
print(f"Test  MSE: {test_mse:.3f}")


In [ ]:
coeffs = shred.latent_forecaster.model.coefficients()
print("coeffs shape:", coeffs.shape)
#print("feature names:", shred.latent_forecaster.model.get_feature_names())
#print(coeffs)

feature_names = shred.latent_forecaster.model.get_feature_names()
# Find which columns correspond to state variables (not '1')
state_cols = [i for i, name in enumerate(feature_names) if name != '1']
A = coeffs[:, state_cols]  # now square (n x n)

eigenvalues = np.linalg.eig(A)[0]
print("Eigenvalues:", eigenvalues)
print("Real parts:", eigenvalues.real)

In [ ]:
engine = SHREDEngine(manager, shred)


In [ ]:
# obtain latent space of test sensor measurements
test_latent_from_sensors = engine.sensor_to_latent(manager.test_sensor_measurements)


In [ ]:
# generate latent states from validation sensor measurements
val_latents = engine.sensor_to_latent(manager.val_sensor_measurements)

# seed the forecaster with the final `seed_length` latent states from validation
init_latents = val_latents[-shred.latent_forecaster.seed_length:] # seed forecaster with final lag timesteps of latent space from val

# set forecast horizon to match the length of the test dataset
h = len(manager.test_sensor_measurements)

# forecast latent states for the test horizon
test_latent_from_forecaster = engine.forecast_latent(h=h, init_latents=init_latents)

In [ ]:
# decode latent space generated from sensor measurements (generated using engine.sensor_to_latent())
test_reconstruction = engine.decode(test_latent_from_sensors)

# decode latent space generated by the latent forecaster (generated using engine.forecast_latent())
test_forecast = engine.decode(test_latent_from_forecaster)

In [ ]:
test_reconstruction["1dks"].shape, test_forecast["1dks"].shape

In [ ]:
u.shape

In [ ]:
data_to_plot = test_reconstruction["1dks"].T
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t[-data_to_plot.shape[1]:])
levels = np.arange(-3, 3, 0.01)
cs = ax.contourf(xx, tt, data_to_plot.T, cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Reconstruction mode: L = {L}, nu = {nu}")

In [ ]:
data_to_plot = test_forecast["1dks"].T
fig, ax = plt.subplots(figsize=(10,8))

xx, tt = np.meshgrid(x, t[-data_to_plot.shape[1]:])
levels = np.arange(-3, 3, 0.01)
cs = ax.contourf(xx, tt, data_to_plot.T, cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Forecast mode: L = {L}, nu = {nu}")

In [ ]:
ground_truth = data[manager.test_indices]  # data shape: (4000, 1024)

fig, ax = plt.subplots(figsize=(10, 8))

xx, tt = np.meshgrid(x, t[-len(data):][manager.test_indices])
cs = ax.contourf(xx, tt, ground_truth, cmap=cm.inferno)
fig.colorbar(cs)

ax.set_xlabel("x")
ax.set_ylabel("t")
ax.set_title(f"Ground Truth (Test): L = {L}, nu = {nu}")

In [ ]:
# visualize(x, test_forecast["1dks"].T, ground_truth.T, 10, 200,
#           labels=("forecasted", "GT"))

In [ ]:
# visualize(x, test_reconstruction["1dks"].T, ground_truth.T, 10, 200,
#           labels=("reconstructed", "GT"))

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPImage

def save_animation(x, u, path, u2=None, labels=('u1', 'u2'), fps=30):
    n_time = u.shape[1]
    ymin = min(u.min(), u2.min()) if u2 is not None else u.min()
    ymax = max(u.max(), u2.max()) if u2 is not None else u.max()
    margin = (ymax - ymin) * 0.1

    fig, ax = plt.subplots(figsize=(10, 6))
    line1, = ax.plot([], [], 'b', linewidth=2, label=labels[0])
    lines = [line1]
    if u2 is not None:
        line2, = ax.plot([], [], 'r', linewidth=2, label=labels[1])
        lines.append(line2)
        ax.legend()
    ax.set_xlim(x[0], x[-1])
    ax.set_ylim(ymin - margin, ymax + margin)
    ax.set_xlabel('Position (x)')
    ax.set_ylabel('Amplitude (u)')
    ax.grid(True, alpha=0.3)
    title = ax.set_title('')

    def update(frame):
        line1.set_data(x, u[:, frame])
        if u2 is not None:
            lines[1].set_data(x, u2[:, frame])
        title.set_text(f'Frame {frame} ({frame/(n_time-1)*100:.1f}%)')
        return lines + [title]

    anim = FuncAnimation(fig, update, frames=n_time, blit=True)
    anim.save(path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    print(f'Saved to {path}')
    return IPImage(filename=path)


In [ ]:
save_animation(x, test_reconstruction["1dks"].T, "reconstructed_vs_gt.gif",
               u2=ground_truth.T, labels=("Reconstructed", "GT"), fps=30)

In [ ]:
save_animation(x, test_forecast["1dks"].T, "forecast_vs_gt.gif",
               u2=ground_truth.T, labels=("Forecast", "GT"), fps=30)